In [57]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/processed_data.csv",
    parse_dates=["Datetime"]
)

df.head()

,Date,OREB [Time unit from - to],Forecasted Total Load [MW],Actual Total Load [MW],Time,Datetime,Hour,Weekday,Month,Month_Name,Time_Difference
0,2026-01-19,00:00 - 00:15,17200,17909.328,00:00,2026-01-19 00:00:00,0,Monday,1,January,NaN
1,2026-01-19,00:15 - 00:30,17050,17439.410,00:15,2026-01-19 00:15:00,0,Monday,1,January,0 days 00:15:00
2,2026-01-19,00:30 - 00:45,16900,17515.727,00:30,2026-01-19 00:30:00,0,Monday,1,January,0 days 00:15:00
3,2026-01-19,00:45 - 01:00,16800,17493.357,00:45,2026-01-19 00:45:00,0,Monday,1,January,0 days 00:15:00
4,2026-01-19,01:00 - 01:15,16700,17276.377,01:00,2026-01-19 01:00:00,1,Monday,1,January,0 days 00:15:00


In [58]:
print(df.shape)
df.info()

(13436, 11)
<class 'pandas.DataFrame'>
RangeIndex: 13436 entries, 0 to 13435
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Date                        13436 non-null  str           
 1   OREB [Time unit from - to]  13436 non-null  str           
 2   Forecasted Total Load [MW]  13436 non-null  int64         
 3   Actual Total Load [MW]      13436 non-null  float64       
 4   Time                        13436 non-null  str           
 5   Datetime                    13436 non-null  datetime64[us]
 6   Hour                        13436 non-null  int64         
 7   Weekday                     13436 non-null  str           
 8   Month                       13436 non-null  int64         
 9   Month_Name                  13436 non-null  str           
 10  Time_Difference             13435 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(3), str(6)
me

In [59]:
for column in df.columns:
    print(column)

Date
OREB [Time unit from - to]
Forecasted Total Load [MW]
Actual Total Load [MW]
Time
Datetime
Hour
Weekday
Month
Month_Name
Time_Difference


In [60]:
actual_column = "Actual Total Load [MW]"
forecast_column = "Forecasted Total Load [MW]"

In [61]:
required_columns = [
    "Datetime",
    "Hour",
    "Weekday",
    "Month",
    actual_column,
    forecast_column
]

for column in required_columns:
    print(f"{column} - {column in df.columns}")

Datetime - True
Hour - True
Weekday - True
Month - True
Actual Total Load [MW] - True
Forecasted Total Load [MW] - True


In [62]:
df["Forecast_Error_MW"] = (
        df[actual_column] - df[forecast_column]
)

df["Absolute_Error_MW"] = (
    df["Forecast_Error_MW"].abs()
)

df["Squared_Error_MW2"] = (
        df["Forecast_Error_MW"] ** 2
)

df["Absolute_Percentage_Error"] = (
        df["Absolute_Error_MW"] / df[actual_column] * 100
)

In [63]:
import numpy as np

df["Forecast_Status"] = np.where(
    df["Forecast_Error_MW"] > 0,
    "Underforecast",
    np.where(
        df["Forecast_Error_MW"] < 0,
        "Overforecast",
        "Exact"
    )
)

In [64]:
df[
    [
        "Datetime",
        actual_column,
        forecast_column,
        "Forecast_Error_MW",
        "Absolute_Error_MW",
        "Absolute_Percentage_Error",
        "Forecast_Status"
    ]
].head()

,Datetime,Actual Total Load [MW],Forecasted Total Load [MW],Forecast_Error_MW,Absolute_Error_MW,Absolute_Percentage_Error,Forecast_Status
0,2026-01-19 00:00:00,17909.328,17200,709.328,709.328,3.960662,Underforecast
1,2026-01-19 00:15:00,17439.410,17050,389.410,389.410,2.232931,Underforecast
2,2026-01-19 00:30:00,17515.727,16900,615.727,615.727,3.515281,Underforecast
3,2026-01-19 00:45:00,17493.357,16800,693.357,693.357,3.963545,Underforecast
4,2026-01-19 01:00:00,17276.377,16700,576.377,576.377,3.336215,Underforecast


In [65]:
df["Year"] = df["Datetime"].dt.year
df["Week"] = df["Datetime"].dt.isocalendar().week.astype(int)
df["Weekday_Number"] = df["Datetime"].dt.dayofweek

df["Is_Weekend"] = df["Weekday_Number"].isin([5, 6])

In [66]:
columns_to_check = [
    actual_column,
    forecast_column,
    "Forecast_Error_MW",
    "Absolute_Error_MW",
    "Squared_Error_MW2",
    "Absolute_Percentage_Error",
    "Forecast_Status"
]

df[columns_to_check].isna().sum()   # checking if there are missing values, if 0 then not

Actual Total Load [MW]        0
Forecasted Total Load [MW]    0
Forecast_Error_MW             0
Absolute_Error_MW             0
Squared_Error_MW2             0
Absolute_Percentage_Error     0
Forecast_Status               0
dtype: int64

In [67]:
df["Year"] = df["Datetime"].dt.year
df["Week"] = df["Datetime"].dt.isocalendar().week.astype(int)
df["Weekday_Number"] = df["Datetime"].dt.dayofweek
df["Is_Weekend"] = df["Weekday_Number"].isin([5, 6])

In [68]:
mae = df["Absolute_Error_MW"].mean()
rmse = np.sqrt(df["Squared_Error_MW2"].mean())
mape = df["Absolute_Percentage_Error"].mean()

print(f"MAE: {mae:.2f} MW")
print(f"RMSE: {rmse:.2f} MW")
print(f"MAPE: {mape:.2f}%")

MAE: 491.55 MW
RMSE: 646.44 MW
MAPE: 2.68%


In [69]:
df = df.rename(columns={
    "Date": "date",
    "Time": "time",
    "Datetime": "datetime",
    "Hour": "hour",
    "Weekday": "weekday",
    "Weekday_Number": "weekday_number",
    "Month": "month",
    "Year": "year",
    "Week": "week",
    "Is_Weekend": "is_weekend",
    "Actual Total Load [MW]": "actual_load_mw",
    "Forecasted Total Load [MW]": "forecasted_load_mw",
    "Forecast_Error_MW": "forecast_error_mw",
    "Absolute_Error_MW": "absolute_error_mw",
    "Squared_Error_MW2": "squared_error_mw2",
    "Absolute_Percentage_Error": "absolute_percentage_error",
    "Forecast_Status": "forecast_status"
})

In [70]:
df.to_csv(
    "../data/processed/load_data_final.csv",
    index=False
)

In [71]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(override=True)

username = os.getenv("POSTGRES_USER", "postgres")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST", "localhost")
port = os.getenv("POSTGRES_PORT", "5432")

# Temporary connection to the default PostgreSQL database
database = "postgres"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user;")
    )
    print(result.fetchone())

('postgres', 'postgres')


In [72]:
with engine.connect() as connection:
    database_exists = connection.execute(
        text("""
            SELECT 1
            FROM pg_database
            WHERE datname = 'energy_market_board';
        """)
    ).scalar()

print(database_exists)

1


In [74]:
with engine.connect() as connection:
    database_exists = connection.execute(
        text("""
            SELECT 1
            FROM pg_database
            WHERE datname = 'energy_market_board';
        """)
    ).scalar()

if not database_exists:
    with engine.connect().execution_options(
        isolation_level="AUTOCOMMIT"
    ) as connection:
        connection.execute(
            text("CREATE DATABASE energy_market_board;")
        )

    print("Database created.")
else:
    print("Database already exists.")

Database already exists.


In [75]:
database = "energy_market_board"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user;")
    )
    print(result.fetchone())

    table_name = "fact_power_system_load"

df.to_sql(
    name=table_name,
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

print(
    f"Data successfully loaded into '{table_name}' "
    f"in database '{database}'."
)

('energy_market_board', 'postgres')
Data successfully loaded into 'fact_power_system_load' in database 'energy_market_board'.
